In [ ]:
# Cell 1: Imports and setup
from common_imports import *
from common_utils import evaluate, train_model, setup_data
from torch.optim.lr_scheduler import StepLR

# 2) Model definition (bigger network + BatchNorm + Dropout)
class SimpleCNN(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super().__init__()
     # Block 1: Input 28x28x1 -> Output 14x14x32
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        # Block 2: Input 14x14x32 -> Output 7x7x64
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # Block 3: Fully Connected
        self.fc = nn.Sequential(
            nn.Linear(64 * 6 * 6, 600),
            nn.Dropout(0.25), # Prevent overfitting on cloth patterns
            nn.ReLU(),
            nn.Linear(600, 120),
            nn.ReLU(),
            nn.Linear(120, 10)
        )
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


def init_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)


# 3) Data setup with augmentation
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.1307), (0.3081)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307), (0.3081)),
])

full_train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=train_transform)
val_size = int(len(full_train_dataset) * 0.1)
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=test_transform), batch_size=128, shuffle=False, num_workers=0, pin_memory=False)

# 4) Training the model
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = SimpleCNN().to(device)
model.apply(init_weights)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)


scheduler = StepLR(optimizer, step_size=7, gamma=0.5)

print("=== Training the model ===")
history = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs=NUM_EPOCHS,
    patience=4,
)

# 5) Evaluate the model
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\nFinal Test Loss: {test_loss:.4f} | Final Test Accuracy: {test_acc:.4f}")

=== Training the model ===
Epoch 01/15 | Train Loss: 0.5930 | Train Acc: 0.7896 | Val Loss: 0.3860 | Val Acc: 0.8507
Epoch 02/15 | Train Loss: 0.3903 | Train Acc: 0.8565 | Val Loss: 0.3431 | Val Acc: 0.8692
Epoch 03/15 | Train Loss: 0.3436 | Train Acc: 0.8732 | Val Loss: 0.3255 | Val Acc: 0.8723
Epoch 04/15 | Train Loss: 0.3098 | Train Acc: 0.8856 | Val Loss: 0.3038 | Val Acc: 0.8815
Epoch 05/15 | Train Loss: 0.2952 | Train Acc: 0.8911 | Val Loss: 0.2842 | Val Acc: 0.8977
Epoch 06/15 | Train Loss: 0.2767 | Train Acc: 0.8990 | Val Loss: 0.2646 | Val Acc: 0.9005
Epoch 07/15 | Train Loss: 0.2616 | Train Acc: 0.9011 | Val Loss: 0.2530 | Val Acc: 0.9068
Epoch 08/15 | Train Loss: 0.2514 | Train Acc: 0.9082 | Val Loss: 0.2415 | Val Acc: 0.9118
Epoch 09/15 | Train Loss: 0.2389 | Train Acc: 0.9114 | Val Loss: 0.2479 | Val Acc: 0.9112
Epoch 10/15 | Train Loss: 0.2340 | Train Acc: 0.9132 | Val Loss: 0.2289 | Val Acc: 0.9135
Epoch 11/15 | Train Loss: 0.2226 | Train Acc: 0.9176 | Val Loss: 0.2307 |